# GroupDNA — WhatsApp Group Chat Analytics
**Project:** GroupDNA — your WhatsApp group chat, decoded  
**Student name:** [N.KARTHIKEYA]  

**Date:** [25-11-2026]

This notebook analyzes the supplied `hostel_bois.txt` WhatsApp export. It produces a readable group report covering message activity, word usage, hourly patterns, response gaps, silent streaks and personality archetypes.

> **Dataset:**  `hostel_bois.txt` 

## Setup and imports
Only NumPy and Python's standard `datetime` module are used for analysis. The parser and counters are written with basic Python data structures. No pandas, plotting libraries, regular expressions, `Counter`, or WhatsApp-analysis packages are used.

In [ ]:
import os
import numpy as np
from datetime import datetime, timedelta

GROUP_NAME = "Hostel Bois 4ever"
DATA_FILE = "hostel_bois.txt"


if not os.path.exists(DATA_FILE):
    possible_paths = ["/content/hostel_bois.txt", "/mnt/data/hostel_bois.txt"]
    for candidate in possible_paths:
        if os.path.exists(candidate):
            DATA_FILE = candidate
            break

print("Dataset path:", DATA_FILE)
print("File found:", os.path.exists(DATA_FILE))

Dataset path: hostel_bois.txt
File found: True


## Feature 1 — Read and parse the WhatsApp export
The parser separates timestamp, sender and message text. It also handles system lines, media placeholders, deleted messages, blank lines and continuation lines. Media and deleted entries remain part of message activity, but are excluded from text-based word and length analysis.

In [ ]:

def looks_like_timestamp(line):
  
    return (len(line) >= 8 and line[0:2].isdigit()
            and line[2] == "/" and line[3:5].isdigit()
            and line[5] == "/" and line[6:8].isdigit())

def parse_chat_file(path):
    messages = []
    system_count = 0
    skipped_lines = 0

    with open(path, "r", encoding="utf-8") as chat_file:
        raw_lines = chat_file.read().splitlines()

    for raw_line in raw_lines:
        line = raw_line.strip()
        if line == "":
            skipped_lines += 1
            continue

        if not looks_like_timestamp(line):
            
            if messages:
                messages[-1]["text"] += " " + line
            else:
                skipped_lines += 1
            continue

        if " - " not in line:
            skipped_lines += 1
            continue

        timestamp_text, remaining = line.split(" - ", 1)

        if ": " not in remaining:
            system_count += 1
            continue

        sender, message_text = remaining.split(": ", 1)
        sender = sender.strip()
        message_text = message_text.strip()

        try:
            parsed_time = datetime.strptime(timestamp_text, "%d/%m/%y, %H:%M")
        except ValueError:
            skipped_lines += 1
            continue

        messages.append({
            "timestamp": timestamp_text,
            "datetime": parsed_time,
            "sender": sender,
            "text": message_text
        })

    return messages, system_count, skipped_lines

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "hostel_bois.txt was not found. Place it beside this notebook or upload it to Colab."
    )

messages, system_message_count, skipped_line_count = parse_chat_file(DATA_FILE)
participants = sorted({message["sender"] for message in messages})
print("Parsed chat entries:", len(messages))
print("Participants:", participants)
print("System messages:", system_message_count)
print("Other skipped lines:", skipped_line_count)
print("\nFirst 3 parsed entries:")
for item in messages[:3]:
    print(item["timestamp"], "-", item["sender"] + ":", item["text"])
print("\nLast 3 parsed entries:")
for item in messages[-3:]:
    print(item["timestamp"], "-", item["sender"] + ":", item["text"])

Parsed chat entries: 3174
Participants: ['Aman', 'Karan', 'Neha', 'Priya', 'Rahul', 'Vikas']
System messages: 4
Other skipped lines: 0

First 3 parsed entries:
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan
01/04/24, 01:18 - Rahul: kya scene

Last 3 parsed entries:
30/05/24, 21:17 - Aman: the existential dread is back
30/05/24, 21:30 - Karan: Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ran around campus collecting signatures for the project document, ate lunch at four PM standing in the corridor, came back to hostel and realized I forgot my charger in the lab.
30/05/24, 23:31 - Aman: anyone awake?


## Feature 2 — Group overview and member activity
Every sender-authored entry contributes to message totals, including media and deleted placeholders. This keeps activity counts consistent with the export. Text statistics use only ordinary text messages.

In [3]:
MEDIA_TEXT = "<Media omitted>"
DELETED_TEXT = "This message was deleted"

media_count_by_person = {person: 0 for person in participants}
deleted_count_by_person = {person: 0 for person in participants}
message_count_by_person = {person: 0 for person in participants}
text_messages = []

for message in messages:
    person = message["sender"]
    message_count_by_person[person] += 1
    if message["text"] == MEDIA_TEXT:
        media_count_by_person[person] += 1
    elif message["text"] == DELETED_TEXT:
        deleted_count_by_person[person] += 1
    else:
        text_messages.append(message)

all_times = [message["datetime"] for message in messages]
first_time = min(all_times)
last_time = max(all_times)
period_days = (last_time.date() - first_time.date()).days + 1
total_messages = len(messages)

print("=" * 68)
print("GROUPDNA REPORT —", GROUP_NAME)
print(f"{period_days} days | {total_messages:,} messages | {len(participants)} members")
print("=" * 68)
print("Period:", first_time.strftime("%d %B %Y"), "to", last_time.strftime("%d %B %Y"))
print("System messages skipped:", system_message_count)
print("Media entries:", sum(media_count_by_person.values()))
print("Deleted entries:", sum(deleted_count_by_person.values()))
print("\nMESSAGES PER PERSON")
for person, count in sorted(message_count_by_person.items(),
                            key=lambda item: item[1], reverse=True):
    percentage = count / total_messages * 100 if total_messages else 0
    bar = "█" * int(25 * count / max(message_count_by_person.values()))
    print(f"{person:<10} {bar:<25} {count:>5} ({percentage:4.1f}%)")
print("\nMEDIA AND DELETED ENTRIES")
for person in participants:
    print(f"{person:<10} media={media_count_by_person[person]:>3} | deleted={deleted_count_by_person[person]:>3}")

GROUPDNA REPORT — Hostel Bois 4ever
60 days | 3,174 messages | 6 members
Period: 01 April 2024 to 30 May 2024
System messages skipped: 4
Media entries: 32
Deleted entries: 15

MESSAGES PER PERSON
Rahul      █████████████████████████   953 (30.0%)
Priya      ██████████████████          718 (22.6%)
Neha       ████████████████            635 (20.0%)
Aman       ████████████                490 (15.4%)
Karan      █████████                   354 (11.2%)
Vikas                                   24 ( 0.8%)

MEDIA AND DELETED ENTRIES
Aman       media=  4 | deleted=  2
Karan      media=  7 | deleted=  2
Neha       media=  8 | deleted=  3
Priya      media=  4 | deleted=  2
Rahul      media=  7 | deleted=  6
Vikas      media=  2 | deleted=  0


## Feature 3 — Busiest day and hour
The busiest day is the calendar date with the most sender-authored entries. The busiest hour is the hour with the largest total across the complete chat.

In [4]:
messages_per_day = {}
messages_per_hour = {hour: 0 for hour in range(24)}

for message in messages:
    day_key = message["datetime"].date()
    hour_key = message["datetime"].hour
    messages_per_day[day_key] = messages_per_day.get(day_key, 0) + 1
    messages_per_hour[hour_key] += 1

busiest_day = max(messages_per_day, key=messages_per_day.get)
busiest_hour = max(messages_per_hour, key=messages_per_hour.get)

print("Busiest day :", busiest_day.strftime("%d %B %Y"),
      f"({messages_per_day[busiest_day]} messages)")
print("Busiest hour:", f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00",
      f"({messages_per_hour[busiest_hour]} messages)")

Busiest day : 04 May 2024 (76 messages)
Busiest hour: 18:00 - 19:00 (248 messages)


## Feature 4 — Activity heatmap using NumPy
A NumPy matrix stores message counts by participant and hour. The terminal-style heatmap uses four intensity levels, as requested in the brief.

In [5]:
person_index = {person: index for index, person in enumerate(participants)}
hour_matrix = np.zeros((len(participants), 24), dtype=int)

for message in messages:
    row = person_index[message["sender"]]
    column = message["datetime"].hour
    hour_matrix[row, column] += 1

print("ACTIVITY HEATMAP — messages by hour")
print("          " + " ".join(f"{hour:02d}" for hour in range(24)))
for row_index, person in enumerate(participants):
    row = hour_matrix[row_index]
    row_max = int(np.max(row)) if len(row) else 0
    symbols = []
    for value in row:
        ratio = value / row_max if row_max else 0
        if value == 0:
            symbols.append("· ")
        elif ratio <= 0.25:
            symbols.append("░ ")
        elif ratio <= 0.50:
            symbols.append("▒ ")
        else:
            symbols.append("█ ")
    print(f"{person:<9}" + "".join(symbols))
print("\nNumPy row totals:", dict(zip(participants, np.sum(hour_matrix, axis=1).tolist())))

ACTIVITY HEATMAP — messages by hour
          00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman     █ █ █ █ █ · · · · · · · · · ░ ░ ░ ░ ░ ░ ░ ░ · █ 
Karan    · · · · · · · ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░ 
Neha     · · · · · ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒ 
Priya    · · · · · · ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░ 
Rahul    ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █ 
Vikas    · · · · · · · ▒ █ ▒ ▒ · █ █ · ▒ ▒ █ █ █ ▒ ▒ ▒ █ 

NumPy row totals: {'Aman': 490, 'Karan': 354, 'Neha': 635, 'Priya': 718, 'Rahul': 953, 'Vikas': 24}


## Feature 5 — Favourite words
Words are converted to lowercase and punctuation is stripped. Common English/Hinglish filler words are ignored. Media and deleted placeholders are not included.

In [6]:
stop_words = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of", "in", "on",
    "for", "it", "you", "me", "my", "we", "are", "am", "was", "were",
    "be", "this", "that", "what", "who", "with", "so", "just", "have",
    "has", "had", "do", "did", "does", "he", "she", "they", "them",
    "your", "our", "at", "as", "but", "if", "not", "no", "yes", "ok",
    "how", "about", "hai", "today", "his", "which", "everyone", "telling", "from", "up",
    "okay", "please", "guys"
}
punctuation = ".,!?;:'\"()[]{}<>-_/\\|@#$%^&*+=~`"

word_counts = {}
words_by_person = {person: {} for person in participants}
word_total_by_person = {person: 0 for person in participants}
message_word_lengths = {person: [] for person in participants}

for message in text_messages:
    person = message["sender"]
    tokens = message["text"].lower().split()
    clean_words = []
    for token in tokens:
        word = token.strip(punctuation)
        if word and word not in stop_words:
            clean_words.append(word)
            word_counts[word] = word_counts.get(word, 0) + 1
            words_by_person[person][word] = words_by_person[person].get(word, 0) + 1
    message_word_lengths[person].append(len(tokens))
    word_total_by_person[person] += len(tokens)

top_words = sorted(word_counts.items(), key=lambda item: item[1], reverse=True)[:10]
print("THIS GROUP'S FAVOURITE WORDS")
largest_word_count = top_words[0][1] if top_words else 1
for word, count in top_words:
    bar = "█" * int(25 * count / largest_word_count)
    print(f"{word:<14} {bar:<25} {count}")

print("\nTOP WORDS BY MEMBER")
for person in participants:
    member_top = sorted(words_by_person[person].items(),
                        key=lambda item: item[1], reverse=True)[:5]
    print(f"{person:<10} {member_top}")

THIS GROUP'S FAVOURITE WORDS
bhai           █████████████████████████ 160
one            ████████████████████████  157
started        ███████████████████████   150
scene          ██████████████████████    145
entire         ██████████████████████    145
anyone         █████████████████████     139
yaar           █████████████████████     139
kya            ████████████████████      133
now            ██████████████████        121
everything     ██████████████████        121

TOP WORDS BY MEMBER
Aman       [('sleep', 71), ("i'm", 56), ('anyone', 49), ('3', 45), ('wonder', 43)]
Karan      [('started', 150), ('entire', 145), ('one', 127), ('came', 116), ('three', 97)]
Neha       [('cant', 58), ('way', 48), ('wait', 48), ('believe', 47), ('literally', 40)]
Priya      [('aman', 93), ('anyone', 90), ('take', 72), ('eat', 60), ('drink', 58)]
Rahul      [('bhai', 159), ('scene', 144), ('kya', 133), ('yaar', 105), ('ja', 101)]
Vikas      [('haha', 4), ('sorry', 3), ('busy', 3), ('tha', 3), ('ha

## Feature 6 — Response gaps and silent streaks
For response-time analysis, a reply is counted when a message is followed by a message from a different participant. The time gap is credited to the sender of that next message. Silent streaks are calculated across every calendar date in the chat period, including dates on which a person sent no messages.

In [7]:
response_gaps = {person: [] for person in participants}
ordered_messages = sorted(messages, key=lambda item: item["datetime"])

for index in range(1, len(ordered_messages)):
    previous_message = ordered_messages[index - 1]
    current_message = ordered_messages[index]
    if current_message["sender"] != previous_message["sender"]:
        gap_minutes = (current_message["datetime"] - previous_message["datetime"]).total_seconds() / 60
        if gap_minutes >= 0:
            response_gaps[current_message["sender"]].append(gap_minutes)

average_response_minutes = {}
for person in participants:
    gaps = response_gaps[person]
    average_response_minutes[person] = (sum(gaps) / len(gaps)) if gaps else None

start_date = first_time.date()
end_date = last_time.date()
all_dates = []
current_date = start_date
while current_date <= end_date:
    all_dates.append(current_date)
    current_date += timedelta(days=1)

active_dates = {person: set() for person in participants}
for message in messages:
    active_dates[message["sender"]].add(message["datetime"].date())

silent_streaks = {}
silent_streak_ranges = {}
for person in participants:
    longest = 0
    current_streak = 0
    longest_start = None
    longest_end = None
    streak_start = None

    for day in all_dates:
        if day not in active_dates[person]:
            if current_streak == 0:
                streak_start = day
            current_streak += 1
            if current_streak > longest:
                longest = current_streak
                longest_start = streak_start
                longest_end = day
        else:
            current_streak = 0
            streak_start = None

    silent_streaks[person] = longest
    silent_streak_ranges[person] = (longest_start, longest_end)

valid_response = [(person, value) for person, value in average_response_minutes.items()
                  if value is not None]
if valid_response:
    fastest = min(valid_response, key=lambda item: item[1])
    slowest = max(valid_response, key=lambda item: item[1])
    print(f"Fastest replier: {fastest[0]} (average {fastest[1]:.1f} minutes)")
    print(f"Slowest replier: {slowest[0]} (average {slowest[1]/60:.1f} hours)")
else:
    print("Not enough cross-person replies to calculate response gaps.")

print("\nLONGEST SILENT STREAKS")
for person in sorted(participants, key=lambda name: silent_streaks[name], reverse=True):
    count = silent_streaks[person]
    date_range = silent_streak_ranges[person]
    if count and date_range[0] is not None:
        range_text = f" ({date_range[0].strftime('%d %b')} to {date_range[1].strftime('%d %b')})"
    else:
        range_text = " (no silent days)" if count == 0 else ""
    print(f"{person:<10}: {count:>2} days{range_text}")

Fastest replier: Rahul (average 34.9 minutes)
Slowest replier: Aman (average 0.9 hours)

LONGEST SILENT STREAKS
Vikas     : 11 days (23 Apr to 03 May)
Aman      :  0 days (no silent days)
Karan     :  0 days (no silent days)
Neha      :  0 days (no silent days)
Priya     :  0 days (no silent days)
Rahul     :  0 days (no silent days)


## Feature 7 — Personality archetypes
Each rule produces a measurable score. The primary six archetypes follow the thresholds in the assignment brief; comedian and question-master scores act as fallbacks. To keep the assignment exclusive, each archetype can be assigned to only one participant: eligible person–archetype pairs are ranked by score, then assigned greedily. Ties are resolved alphabetically by participant name. These are playful chat-pattern labels, not psychological assessments.

In [ ]:

def get_person_metrics(person):
    own_messages = [message for message in messages if message["sender"] == person]
    own_text = [message for message in own_messages
                if message["text"] not in (MEDIA_TEXT, DELETED_TEXT)]
    count = len(own_messages)
    night_count = sum(1 for message in own_messages
                      if message["datetime"].hour >= 23 or message["datetime"].hour <= 4)
    all_caps_count = 0
    exclamation_count = 0
    question_count = 0
    caring_count = 0
    comedian_count = 0
    burst_lengths = []
    current_burst = 0

    caring_terms = ["okay", "safe", "eat", "sleep", "take care", "are you",
                    "please", "reminder", "drink water", "don't forget"]
    funny_terms = ["lol", "lmao", "haha", "rofl", "lmfao"]

    for message in own_messages:
        content = message["text"]
        lower_content = content.lower()
        if content not in (MEDIA_TEXT, DELETED_TEXT):
            letters = [character for character in content if character.isalpha()]
            is_all_caps = len(letters) >= 3 and content == content.upper()
            has_multiple_exclamations = content.count("!") >= 2
            if is_all_caps or has_multiple_exclamations:
                all_caps_count += 1
            if content.rstrip().endswith("?"):
                question_count += 1
            if any(term in lower_content for term in caring_terms):
                caring_count += 1
            for term in funny_terms:
                if term in lower_content:
                    comedian_count += 1

    for index, message in enumerate(ordered_messages):
        if message["sender"] == person:
            if index == 0 or ordered_messages[index - 1]["sender"] == person:
                current_burst += 1
            else:
                if current_burst > 1:
                    burst_lengths.append(current_burst)
                current_burst = 1
        elif current_burst:
            if current_burst > 1:
                burst_lengths.append(current_burst)
            current_burst = 0
    if current_burst > 1:
        burst_lengths.append(current_burst)

    average_burst = (sum(burst_lengths) / len(burst_lengths)) if burst_lengths else 0
    average_words = (sum(len(item["text"].split()) for item in own_text) / len(own_text)
                     if own_text else 0)
    silent_ratio = ((period_days - len(active_dates[person])) / period_days
                    if period_days else 0)
    return {
        "count": count,
        "night_ratio": night_count / count if count else 0,
        "average_burst": average_burst,
        "average_words": average_words,
        "drama_ratio": (all_caps_count + exclamation_count) / count if count else 0,
        "silent_ratio": silent_ratio,
        "caring_count": caring_count,
        "comedian_ratio": comedian_count / count if count else 0,
        "question_ratio": question_count / count if count else 0
    }

metrics = {person: get_person_metrics(person) for person in participants}


candidate_pairs = []
for person, values in metrics.items():
    if values["average_burst"] > 3:
        candidate_pairs.append((values["average_burst"] / 3, person, "THE SPAMMER",
                                f"average burst {values['average_burst']:.1f}"))
    if values["caring_count"] > 0:
        candidate_pairs.append((values["caring_count"], person, "THE GROUP MOM",
                                f"{values['caring_count']} caring keyword matches"))
    if values["night_ratio"] > 0.60:
        candidate_pairs.append((values["night_ratio"] / 0.60, person, "THE NIGHT OWL",
                                f"{values['night_ratio']*100:.1f}% of messages from 23:00–04:59"))
    if values["average_words"] > 30:
        candidate_pairs.append((values["average_words"] / 30, person, "THE STORYTELLER",
                                f"{values['average_words']:.1f} words per text message"))
    if values["drama_ratio"] > 0.30:
        candidate_pairs.append((values["drama_ratio"] / 0.30, person, "THE DRAMA QUEEN",
                                f"{values['drama_ratio']*100:.1f}% drama-style messages"))
    if values["silent_ratio"] > 0.60:
        candidate_pairs.append((values["silent_ratio"] / 0.60, person, "THE GHOST",
                                f"silent on {values['silent_ratio']*100:.1f}% of days"))

    
    candidate_pairs.append((values["comedian_ratio"], person, "THE COMEDIAN",
                            f"{values['comedian_ratio']*100:.1f}% humour-term rate"))
    candidate_pairs.append((values["question_ratio"], person, "THE QUESTION MASTER",
                            f"{values['question_ratio']*100:.1f}% questions"))

archetype_priority = [
    "THE NIGHT OWL", "THE GROUP MOM", "THE STORYTELLER",
    "THE DRAMA QUEEN", "THE GHOST", "THE SPAMMER",
    "THE COMEDIAN", "THE QUESTION MASTER"
]
assigned_people = set()
assigned_archetypes = set()
archetype_assignments = {}

for archetype in archetype_priority:
    eligible = [item for item in candidate_pairs if item[2] == archetype
                and item[1] not in assigned_people]
    if eligible:
        eligible.sort(key=lambda item: (-item[0], item[1]))
        score, person, label, explanation = eligible[0]
        archetype_assignments[person] = (label, explanation, score)
        assigned_people.add(person)
        assigned_archetypes.add(label)


for person in participants:
    if person not in archetype_assignments:
        available = [item for item in candidate_pairs
                     if item[1] == person and item[2] not in assigned_archetypes]
        if available:
            available.sort(key=lambda item: (-item[0], item[2]))
            score, _, archetype, explanation = available[0]
            archetype_assignments[person] = (archetype, explanation, score)
            assigned_archetypes.add(archetype)

print("PERSONALITY ARCHETYPES")
for person in participants:
    if person in archetype_assignments:
        archetype, explanation, score = archetype_assignments[person]
        print(f"{person:<10} → {archetype:<22} ({explanation})")
    else:
        print(f"{person:<10} → UNCLASSIFIED")

PERSONALITY ARCHETYPES
Aman       → THE NIGHT OWL          (79.8% of messages from 23:00–04:59)
Karan      → THE STORYTELLER        (57.0 words per text message)
Neha       → THE DRAMA QUEEN        (62.2% drama-style messages)
Priya      → THE GROUP MOM          (459 caring keyword matches)
Rahul      → THE SPAMMER            (average burst 4.8)
Vikas      → THE GHOST              (silent on 73.3% of days)


## Feature 8 — Final report and quick checks
This final cell brings the main findings together in one console-style summary. Small differences in word counts are expected because stop-word and punctuation choices can vary. The important checks are that all participants are detected, the hourly matrix totals match message totals, and the known behavioural patterns are visible.

In [9]:
print("\n" + "=" * 68)
print("GROUPDNA — FINAL SNAPSHOT")
print("=" * 68)
print(f"Group: {GROUP_NAME}")
print(f"Messages: {total_messages:,} | Members: {len(participants)} | Period: {period_days} days")
print(f"Busiest day: {busiest_day.strftime('%d %B %Y')} ({messages_per_day[busiest_day]} messages)")
print(f"Peak hour: {busiest_hour:02d}:00–{(busiest_hour + 1) % 24:02d}:00")
print("\nMost-used words:")
for word, count in top_words[:5]:
    print(f"  {word:<12} {count}")
print("\nArchetype assignments:")
for person in participants:
    assigned = archetype_assignments.get(person, ("UNCLASSIFIED", "", 0))[0]
    print(f"  {person:<10} {assigned}")
print("\nValidation:")
print("  Matrix total matches parsed messages:", int(np.sum(hour_matrix)) == len(messages))
print("  All parsed senders represented:", set(participants) == set(message_count_by_person))
print("  Parsed message count:", len(messages))
print("  System message count:", system_message_count)
print("=" * 68)


GROUPDNA — FINAL SNAPSHOT
Group: Hostel Bois 4ever
Messages: 3,174 | Members: 6 | Period: 60 days
Busiest day: 04 May 2024 (76 messages)
Peak hour: 18:00–19:00

Most-used words:
  bhai         160
  one          157
  started      150
  scene        145
  entire       145

Archetype assignments:
  Aman       THE NIGHT OWL
  Karan      THE STORYTELLER
  Neha       THE DRAMA QUEEN
  Priya      THE GROUP MOM
  Rahul      THE SPAMMER
  Vikas      THE GHOST

Validation:
  Matrix total matches parsed messages: True
  All parsed senders represented: True
  Parsed message count: 3174
  System message count: 4
